Habitat feature selection now uses whole-image ICC-selected features as the candidate feature pool, removes shape features first, and then reruns PCC separately within each habitat representation (avg/sum/individual).


In [1]:

# ============================================================
# Step 2: Habitat feature selection
# Two modes are supported:
#   1. reuse_whole_pcc: reuse whole-tumor PCC-selected features, then remove shape features.
#   2. remove_shape_then_rerun_pcc: remove shape features first, then rerun PCC inside the current habitat table.
# ============================================================

import os
import numpy as np
import pandas as pd


In [17]:

# ============================================================
# 1. Settings
# ============================================================

# Candidate features come from whole-image ICC output.
# PCC is then recalculated inside each habitat table itself.
whole_icc_path = "/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_ICC.xlsx"

# Change this root to switch among:
#   fixed-K habitats avg/sum: /host/d/projects/Habitats/radiomics/habitats
#   individual-K habitats avg: /host/d/projects/Habitats/radiomics/habitats_individual
habitat_root = "/host/d/projects/Habitats/radiomics/habitats"

# Run both fixed-K habitat representations here.
# For habitats_individual, use modes_to_run = ["avg"].
modes_to_run = ["avg", "sum"]

pcc_threshold = 0.90

non_feature_cols = [
    "Patient_set",
    "Patient_index",
    "Image_filepath",
    "Mask_filepath",
]

print("Whole-image ICC feature source:", whole_icc_path)
print("Habitat root:", habitat_root)
print("Modes to run:", modes_to_run)
print("PCC threshold:", pcc_threshold)


Feature selection mode: remove_shape_then_rerun_pcc
Habitat normalized input: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_radiomics_measurements_avg_normalized.xlsx
Habitat PCC output: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_radiomics_measurements_avg_PCC.xlsx
Feature list output: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_PCC_feature_list_without_shape_avg.xlsx


In [18]:

# ============================================================
# 2. Load whole-image ICC feature pool
# ============================================================

whole_icc_df = pd.read_excel(whole_icc_path)

missing_non_feature_cols = [
    col for col in non_feature_cols
    if col not in whole_icc_df.columns
]
if len(missing_non_feature_cols) > 0:
    raise KeyError(f"Missing non-feature columns in whole ICC table: {missing_non_feature_cols}")

whole_icc_features = [
    col for col in whole_icc_df.columns
    if col not in non_feature_cols
]

whole_icc_shape_features = [
    col for col in whole_icc_features
    if col.startswith("original_shape_")
]

whole_icc_non_shape_features = [
    col for col in whole_icc_features
    if col not in whole_icc_shape_features
]

print("Whole-image ICC table shape:", whole_icc_df.shape)
print("Number of whole-image ICC features:", len(whole_icc_features))
print("Number of shape features removed from whole-image ICC list:", len(whole_icc_shape_features))
print("Number of whole-image ICC non-shape candidate features:", len(whole_icc_non_shape_features))

if len(whole_icc_non_shape_features) == 0:
    raise RuntimeError("No non-shape whole-image ICC features are available for habitat PCC.")


Habitat normalized table shape: (351, 1019)
Number of all habitat features: 1015
Number of shape features removed first: 14
Number of non-shape habitat features: 1001


In [19]:

# ============================================================
# 3. Habitat PCC helper
# ============================================================

def run_habitat_pcc_for_mode(mode):
    input_table_name = f"habitat_radiomics_measurements_{mode}_normalized.xlsx"
    output_table_name = f"habitat_radiomics_measurements_{mode}_PCC.xlsx"
    feature_list_name = f"habitat_PCC_feature_list_without_shape_{mode}.xlsx"

    habitat_normalized_path = os.path.join(habitat_root, input_table_name)
    habitat_pcc_out_path = os.path.join(habitat_root, output_table_name)
    feature_list_out_path = os.path.join(habitat_root, feature_list_name)

    print("\n" + "=" * 80)
    print("Running habitat feature selection mode:", mode)
    print("Habitat normalized input:", habitat_normalized_path)
    print("Habitat PCC output:", habitat_pcc_out_path)
    print("Feature list output:", feature_list_out_path)

    habitat_df = pd.read_excel(habitat_normalized_path)
    print("Habitat normalized table shape:", habitat_df.shape)

    missing_non_feature_cols = [
        col for col in non_feature_cols
        if col not in habitat_df.columns
    ]
    if len(missing_non_feature_cols) > 0:
        raise KeyError(f"Missing non-feature columns in habitat table: {missing_non_feature_cols}")

    all_habitat_features = [
        col for col in habitat_df.columns
        if col not in non_feature_cols
    ]

    habitat_shape_features = [
        col for col in all_habitat_features
        if col.startswith("original_shape_")
    ]

    missing_from_habitat = [
        col for col in whole_icc_non_shape_features
        if col not in habitat_df.columns
    ]

    candidate_features = [
        col for col in whole_icc_non_shape_features
        if col in habitat_df.columns
    ]

    print("Number of all habitat features:", len(all_habitat_features))
    print("Number of habitat shape features present but excluded:", len(habitat_shape_features))
    print("Number of whole-ICC non-shape candidates missing from habitat table:", len(missing_from_habitat))
    print("Number of candidate features entering habitat checks:", len(candidate_features))

    if len(candidate_features) == 0:
        raise RuntimeError("No whole-ICC non-shape candidate features are present in this habitat table.")

    X0 = habitat_df[candidate_features].copy()

    non_numeric_features = [
        col for col in X0.columns
        if not pd.api.types.is_numeric_dtype(X0[col])
    ]

    numeric_features = [
        col for col in X0.columns
        if col not in non_numeric_features
    ]

    X = X0[numeric_features].astype(float)

    all_nan_features = [col for col in X.columns if X[col].isna().all()]
    constant_features = [
        col for col in X.columns
        if col not in all_nan_features and X[col].nunique(dropna=True) <= 1
    ]

    pre_pcc_removed_features = non_numeric_features + all_nan_features + constant_features

    if len(pre_pcc_removed_features) > 0:
        print("Features removed before PCC because they are non-numeric/all-NaN/constant:", len(pre_pcc_removed_features))
        for f in pre_pcc_removed_features[:20]:
            print(" ", f)
        if len(pre_pcc_removed_features) > 20:
            print("  ...")

    X = X.drop(columns=[col for col in all_nan_features + constant_features if col in X.columns])

    if X.shape[1] == 0:
        raise RuntimeError("No numeric nonconstant candidate features left for PCC.")

    corr_matrix = X.corr(method="pearson").abs()
    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    removed_by_pcc = [
        col for col in upper.columns
        if any(upper[col] > pcc_threshold)
    ]

    final_features = [
        col for col in X.columns
        if col not in removed_by_pcc
    ]

    if len(final_features) == 0:
        raise RuntimeError("No final features selected after habitat PCC.")

    print("PCC threshold:", pcc_threshold)
    print("Number of features entering PCC:", X.shape[1])
    print("Number of features removed by PCC:", len(removed_by_pcc))
    print("Number of final habitat features:", len(final_features))

    candidate_status_rows = []
    removed_pre_set = set(pre_pcc_removed_features)
    removed_pcc_set = set(removed_by_pcc)
    final_set = set(final_features)
    missing_set = set(missing_from_habitat)
    shape_set = set(whole_icc_shape_features)

    for f in whole_icc_features:
        if f in shape_set:
            status = "excluded_shape"
        elif f in missing_set:
            status = "excluded_missing_in_habitat"
        elif f in removed_pre_set:
            status = "excluded_pre_pcc"
        elif f in removed_pcc_set:
            status = "excluded_pcc"
        elif f in final_set:
            status = "included"
        else:
            status = "not_used_unknown"
        candidate_status_rows.append({
            "feature_name": f,
            "status": status,
            "included": status == "included",
        })

    candidate_status_df = pd.DataFrame(candidate_status_rows)
    final_features_df = pd.DataFrame({"feature_name": final_features})
    removed_shape_df = pd.DataFrame({"removed_shape_feature": whole_icc_shape_features})
    removed_missing_df = pd.DataFrame({"removed_missing_in_habitat": missing_from_habitat})
    removed_pre_df = pd.DataFrame({"removed_pre_pcc_feature": pre_pcc_removed_features})
    removed_pcc_df = pd.DataFrame({"removed_pcc_feature": removed_by_pcc})

    settings_df = pd.DataFrame([
        {
            "feature_pool_source": "whole_image_ICC_then_remove_shape_then_habitat_PCC",
            "whole_icc_path": whole_icc_path,
            "habitat_normalized_path": habitat_normalized_path,
            "habitat_pcc_out_path": habitat_pcc_out_path,
            "feature_list_out_path": feature_list_out_path,
            "mode": mode,
            "pcc_threshold": pcc_threshold,
            "n_whole_icc_features": len(whole_icc_features),
            "n_removed_shape_features_from_whole_icc": len(whole_icc_shape_features),
            "n_whole_icc_non_shape_candidate_features": len(whole_icc_non_shape_features),
            "n_missing_from_habitat": len(missing_from_habitat),
            "n_features_entering_pcc": X.shape[1],
            "n_removed_pre_pcc_features": len(pre_pcc_removed_features),
            "n_removed_pcc_features": len(removed_by_pcc),
            "n_final_features": len(final_features),
            "n_cases": habitat_df.shape[0],
        }
    ])

    with pd.ExcelWriter(feature_list_out_path, engine="openpyxl") as writer:
        final_features_df.to_excel(writer, sheet_name="final_features", index=False)
        candidate_status_df.to_excel(writer, sheet_name="all_whole_icc_status", index=False)
        removed_shape_df.to_excel(writer, sheet_name="removed_shape", index=False)
        removed_missing_df.to_excel(writer, sheet_name="removed_missing", index=False)
        removed_pre_df.to_excel(writer, sheet_name="removed_pre_pcc", index=False)
        removed_pcc_df.to_excel(writer, sheet_name="removed_pcc", index=False)
        settings_df.to_excel(writer, sheet_name="settings", index=False)

    habitat_pcc_df = habitat_df[non_feature_cols + final_features].copy()
    habitat_pcc_df.to_excel(habitat_pcc_out_path, index=False)

    print("Saved final feature list:", feature_list_out_path)
    print("Saved habitat PCC table:", habitat_pcc_out_path)
    print("Final habitat PCC table shape:", habitat_pcc_df.shape)
    print("Cases by Patient_set:")
    print(habitat_pcc_df["Patient_set"].value_counts().sort_index().to_string())

    return {
        "mode": mode,
        "n_cases": habitat_pcc_df.shape[0],
        "n_final_features": len(final_features),
        "n_features_entering_pcc": X.shape[1],
        "n_removed_pcc_features": len(removed_by_pcc),
        "output_path": habitat_pcc_out_path,
        "feature_list_path": feature_list_out_path,
    }


PCC threshold: 0.9
Number of features entering PCC: 1001
Number of features removed by PCC/pre-PCC checks: 748
Number of final habitat features: 253
All final selected features are present in habitat table.
Final feature number: 253


In [20]:

# ============================================================
# 4. Run selected habitat modes
# ============================================================

results = []
for mode in modes_to_run:
    results.append(run_habitat_pcc_for_mode(mode))

results_df = pd.DataFrame(results)
print("\n" + "=" * 80)
print("Habitat feature-selection summary")
display(results_df)


Saved final feature list: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_PCC_feature_list_without_shape_avg.xlsx
Final habitat PCC table shape: (351, 257)
Saved: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_radiomics_measurements_avg_PCC.xlsx
